<a href="https://colab.research.google.com/github/adya76/dysarthric-speech-detection/blob/main/%20%20%20%20speech_generation/approach_2_spectrogram_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Approach 2: 2D CNN Autoencoder with Griffin-Lim

Mel-spectrogram-based dysarthric speech synthesis using a 2D CNN autoencoder trained on TORGO recordings. The trained model is applied to healthy LibriTTS-R speech, followed by Griffin-Lim waveform reconstruction.

## 1. Imports

In [ ]:
!pip install fastdtw praat-parselmouth

In [ ]:
import subprocess
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torch.nn.functional as F
import glob
from scipy.spatial.distance import cosine
from scipy.spatial.distance import euclidean
import librosa.display
import matplotlib.pyplot as plt
import random
import sklearn.preprocessing
import parselmouth
from parselmouth.praat import call

## 2. Dataset Paths & Hyperparameters

In [ ]:
# Configure dataset paths before running
DATA_ROOT = "/path/to/your/datasets"
TORGO_CSV = os.path.join(DATA_ROOT, "TORGO_processed/train.csv")
TORGO_ROOT = os.path.join(DATA_ROOT, "TORGO")
LIBRITTS_DIR = os.path.join(DATA_ROOT, "LibriTTS-R/LibriTTS_R/dev-clean")
OUTPUT_DIR = "./generated_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Hyperparameters (Optimized for Spectrograms)
SAMPLE_RATE = 16000
N_FFT = 1024       # Resolution of frequency
HOP_LEN = 256      # Resolution of time
N_MELS = 80        # Height of the image (Frequency bands)
FIXED_LEN = 128    # Width of the image (Time frames ~2 seconds)

BATCH_SIZE = 16    # Can handle larger batches with spectrograms
EPOCHS = 10        # Spectrograms need slightly longer to converge
LEARNING_RATE = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Dataset Preparation

In [ ]:
class MelSpectrogramDataset(Dataset):
    def __init__(self, csv_file=None, root_dir=None, file_list=None, mode="train"):
        self.mode = mode
        if mode == "train":
            if os.path.exists(csv_file):
                self.data = pd.read_csv(csv_file)
            else:
                self.data = pd.DataFrame()
            self.root = root_dir
        else:
            self.data = file_list
            self.root = ""

    def __len__(self):
        return len(self.data)

    def load_mel(self, path):
        # 1. Load Audio
        y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
        # 2. Normalize Volume (Crucial for dysarthria)
        if np.max(np.abs(y)) > 0:
            y = y / np.max(np.abs(y))
        # 3. Convert to Mel-Spectrogram
        mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LEN, n_mels=N_MELS)
        # 4. Convert to Log Scale (dB)
        mel_db = librosa.power_to_db(mel, ref=np.max, top_db=80.0)
        # 5. Clip to a fixed dB range
        mel_db = np.clip(mel_db, -80.0, 0.0)
        # 6. Normalize to [-1, 1]
        mel_norm = (mel_db + 80.0) / 80.0
        mel_norm = 2.0 * mel_norm - 1.0
        return torch.tensor(mel_norm, dtype=torch.float32)

    def __getitem__(self, idx):
        try:
            if self.mode == "train":
                rel_path = self.data.iloc[idx, 0]
                wav_path = os.path.join(self.root, rel_path)
            else:
                wav_path = self.data[idx]

            if not os.path.exists(wav_path):
              raise FileNotFoundError(f"Audio file not found: {wav_path}")

            mel_spec = self.load_mel(wav_path)
            original_len = mel_spec.shape[1] # Save the real length!

            if self.mode == "train":
                # --- TRAINING: Random Crop for Batching ---
                if mel_spec.shape[1] > FIXED_LEN:
                    start = np.random.randint(0, mel_spec.shape[1] - FIXED_LEN)
                    mel_spec = mel_spec[:, start : start + FIXED_LEN]
                elif mel_spec.shape[1] < FIXED_LEN:
                    padding = FIXED_LEN - mel_spec.shape[1]
                    mel_spec = F.pad(mel_spec, (0, padding))

                return mel_spec.unsqueeze(0), wav_path, original_len

            else:
                # --- INFERENCE: Keep Full Length ---
                # Math rule: CNN has 3 strides of 2 (2^3 = 8).
                # Width must be a multiple of 8 for the decoder to work.
                remainder = mel_spec.shape[1] % 8
                if remainder != 0:
                    pad_amt = 8 - remainder
                    mel_spec = F.pad(mel_spec, (0, pad_amt))

                return mel_spec.unsqueeze(0), wav_path, original_len

        except Exception as e:
            print(f"Error loading sample {idx}: {e}")
            raise

## 4. CNN Autoencoder

In [ ]:
class SpectrogramAutoencoder(nn.Module):
    def __init__(self):
        super(SpectrogramAutoencoder, self).__init__()

        # ENCODER: Compresses the image (80x128)
        self.encoder = nn.Sequential(
            # Layer 1: Captures fine details (Kernel 3x3)
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # Layer 2: Captures broader shapes (Formants)
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # Layer 3: Bottleneck (Deep Features)
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )

        # DECODER: Reconstructs the image
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh() # Output must be [-1, 1]
        )

    def forward(self, x):
        features = self.encoder(x)
        output = self.decoder(features)
        return output


## 5. Model Training

In [ ]:
train_dataset = MelSpectrogramDataset(csv_file=TORGO_CSV, root_dir=TORGO_ROOT, mode="train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model = SpectrogramAutoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.L1Loss() # Better for images/spectrograms than MSE

print("\ntraining model")
model.train()

for epoch in range(EPOCHS):
    loss_total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for mel, paths, orig_len in pbar:
        if paths[0] == "error": continue

        mel = mel.to(device)

        # Forward
        optimizer.zero_grad()
        output = model(mel)
        loss = criterion(output, mel)

        # Backward
        loss.backward()
        optimizer.step()

        loss_total += loss.item()
        pbar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} done. Avg Loss: {loss_total / len(train_loader):.5f}")

torch.save(model.state_dict(), "spec_cnn_final.pth")
print("Model Saved.")

## 6. Speech Generation using Griffin-Lim

In [ ]:
print("\n Generating Audio (Griffin-Lim)")

libri_files = []
for root, _, files in os.walk(LIBRITTS_DIR):
    for f in files:
        if f.endswith(".wav"): libri_files.append(os.path.join(root, f))

print(f"Found {len(libri_files)} files.")

if len(libri_files) > 0:
    gen_dataset = MelSpectrogramDataset(file_list=libri_files, mode="inf")
    gen_loader = DataLoader(gen_dataset, batch_size=1, shuffle=False)

    model.eval()
    count = 0

    with torch.no_grad():
        for mel, path, orig_len in tqdm(gen_loader, desc="Converting"):
            gen_mel = model(mel.to(device))
            gen_mel = gen_mel.squeeze().cpu().numpy()
            actual_length = orig_len.item()
            gen_mel = gen_mel[:, :actual_length]
            # Convert model output from [-1, 1] back to [0, 1]
            gen_mel_norm = np.clip((gen_mel + 1.0) / 2.0, 0.0, 1.0)
            # Reverse the fixed dB normalisation
            gen_mel_db = gen_mel_norm * 80.0 - 80.0
            # Convert dB values back to power
            gen_mel_pow = librosa.db_to_power(gen_mel_db)

            # 5. VOCODER (Spectrogram -> Audio)
            audio = librosa.feature.inverse.mel_to_audio(gen_mel_pow,sr=SAMPLE_RATE,n_fft=N_FFT,hop_length=HOP_LEN,power=2.0)

            filename = f"spec_dys_{os.path.basename(path[0])}"
            sf.write(os.path.join(OUTPUT_DIR, filename), audio, SAMPLE_RATE)
            count += 1

    print(f"Done. Saved {count} files to {OUTPUT_DIR}")
else:
    print("No LibriTTS files found.")


 Generating Audio (Griffin-Lim)
Found 5744 files.


Converting:   0%|          | 6/5744 [00:03<59:12,  1.62it/s]


KeyboardInterrupt: 

## 7. Acoustic Evaluation

In [ ]:
GEN_DIR = OUTPUT_DIR
TORGO_DIR = TORGO_ROOT
LIBRI_DIR = LIBRITTS_DIR

LIMIT = 50

In [ ]:
def get_acoustics(path):
    try:
        sound = parselmouth.Sound(path)
        pitch = sound.to_pitch()
        pulses = call([sound, pitch], "To PointProcess (cc)")
        f0 = call(pitch, "Get mean", 0, 0, "Hertz")
        formant = sound.to_formant_burg(0.01, 5, 5500)
        f1 = call(formant, "Get mean", 1, 0, 0, "Hertz")
        f2 = call(formant, "Get mean", 2, 0, 0, "Hertz")
        jit = call(pulses, "Get jitter (local)", 0.0, 0.0, 0.0001, 0.02, 1.3)
        shim = call([sound, pulses], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        return np.array([f0, f1, f2, jit, shim])
    except:
        return np.array([np.nan] * 5)

def get_cross_corr(path1, path2):
    try:
        y1, _ = librosa.load(path1, sr=16000)
        y2, _ = librosa.load(path2, sr=16000)
        min_len = min(len(y1), len(y2))
        return np.abs(np.corrcoef(y1[:min_len], y2[:min_len])[0, 1])
    except:
        return np.nan

def get_mcd_dtw(path1, path2):
    try:
        y1, sr = librosa.load(path1, sr=16000)
        y2, _ = librosa.load(path2, sr=16000)

        mfcc1 = librosa.feature.mfcc(y=y1, sr=sr, n_mfcc=13, hop_length=256)
        mfcc2 = librosa.feature.mfcc(y=y2, sr=sr, n_mfcc=13, hop_length=256)

        dist, _ = librosa.sequence.dtw(mfcc1, mfcc2, metric='euclidean')

        return dist[-1, -1] / len(dist)
    except:
        return np.nan


In [ ]:
gen_files = sorted(glob.glob(os.path.join(GEN_DIR, "spec_dys_*.wav")))[:LIMIT]
torgo_files = sorted(glob.glob(f"{TORGO_DIR}/**/*.wav", recursive=True))[:LIMIT]

# 2. Compare Gen vs Real TORGO (Style)
gen_feats = []
torgo_feats = []

for f in tqdm(gen_files, desc="Gen Features"):
    gen_feats.append(get_acoustics(f))

for f in tqdm(torgo_files, desc="TORGO Features"):
    torgo_feats.append(get_acoustics(f))

# Average vectors (ignoring NaNs)
vec_gen = np.nanmean(gen_feats, axis=0)
vec_torgo = np.nanmean(torgo_feats, axis=0)

# 3. Compare Gen vs Original LibriTTS (Content & Spectral)
corrs = []
mcds = []

for gen_path in tqdm(gen_files):
    fname = os.path.basename(gen_path)
    # Adjust replacement based on your filename prefix (dys_, spec_dys_, etc)
    orig_name = os.path.basename(gen_path).replace("spec_dys_", "", 1)
    orig_candidates = glob.glob(f"{LIBRI_DIR}/**/{orig_name}", recursive=True)
    if orig_candidates:
        orig_path = orig_candidates[0]
        # Cross Correlation
        corrs.append(get_cross_corr(gen_path, orig_path))
        # MCD-DTW
        mcds.append(get_mcd_dtw(gen_path, orig_path))

# Cosine Similarity: 1 - Cosine Distance
norm_gen = vec_gen / np.linalg.norm(vec_gen)
norm_torgo = vec_torgo / np.linalg.norm(vec_torgo)
cosine_sim = 1 - cosine(norm_gen, norm_torgo)

print(f"\n1. FEATURE-WISE SIMILARITY (Relative Error %)")
print(f"   (Compare Gen directly to Real TORGO averages)")
feat_names = ["Pitch (F0)", "Formant 1 (F1)", "Formant 2 (F2)", "Jitter", "Shimmer"]
print(f"   {'Feature':<20} | {'Real TORGO':<10} | {'Generated':<10} | {'Similarity':<10}")


for i in range(5):
    real_val = vec_torgo[i]
    gen_val = vec_gen[i]
    sim_score = 100 - (abs(real_val - gen_val) / real_val * 100)

    print(f"   {feat_names[i]:<20} | {real_val:<10.4f} | {gen_val:<10.4f} | {sim_score:.2f}%")


print(f"\n2. Cosine Similarity")
print(f"   Score: {cosine_sim:.4f}")
#print("   (> 0.9 = Excellent Match across all features combined)")

print(f"\n3. SPECTRAL ACCURACY (Gen vs Original)")
print(f"   Avg MCD-DTW Score: {np.nanmean(mcds):.4f}")
#print("   (Lower is Better. < 8.0 is acceptable)")

print(f"\n4. TIMING/CONTENT (Gen vs Original)")
print(f"   Avg Cross-Correlation: {np.nanmean(corrs):.4f}")
#print("   (Higher is Better. Measures content preservation)")


In [ ]:
GENERATED_DIR = OUTPUT_DIR

print(f"Scanning files in: {GENERATED_DIR}")
files = [f for f in os.listdir(GENERATED_DIR) if f.endswith(".wav")]

if not files:
    print("Error: No .wav files found in the directory.")
else:
    print(f"Found {len(files)} files. Calculating Spectral Centroids...")

    results = []

    for wav_file in tqdm(files):
        path = os.path.join(GENERATED_DIR, wav_file)

        try:
            y, sr = librosa.load(path, sr=None)
            # Calculate Centroid
            # Returns an array of frequencies (one per frame)
            centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
            # We take the MEAN to get a single number for the whole file
            avg_centroid = np.mean(centroids)
            std_centroid = np.std(centroids) # Standard deviation (how much it varies)

            results.append({"filename": wav_file,"avg_spectral_centroid": avg_centroid,"std_spectral_centroid": std_centroid})

        except Exception as e:
            print(f"Error processing {wav_file}: {e}")

    df = pd.DataFrame(results)
    df.to_csv("spectral_centroid_analysis.csv", index=False)

    print(f" Total Files Processed: {len(df)}")
    print(f" Mean Centroid (Brightness): {df['avg_spectral_centroid'].mean():.2f} Hz")
    print(f" Std Dev (Variability):      {df['avg_spectral_centroid'].std():.2f} Hz")
    print("Detailed results saved to 'spectral_centroid_analysis.csv'")

    sample_row = df.sample(1).iloc[0]
    sample_path = os.path.join(GENERATED_DIR, sample_row['filename'])

    print(f"\nVisualizing sample: {sample_row['filename']}")

    y, sr = librosa.load(sample_path, sr=None)
    centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    times = librosa.times_like(centroids)

    plt.figure(figsize=(12, 6))

    # Plot Spectrogram
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, y_axis='log', x_axis='time', sr=sr)

    # Plot Centroid Line (Normalized to fit on plot)
    # We normalize it to fit between the min and max frequencies of the plot for visibility
    norm_centroids = sklearn.preprocessing.minmax_scale(centroids, feature_range=(0, sr/2))
    plt.plot(times, centroids, label='Spectral Centroid', color='w', linewidth=2)

    plt.title(f"Spectral Centroid: {sample_row['filename']}")
    plt.legend(loc='upper right')
    plt.colorbar(format='%+2.0f dB')
    plt.show()